# 02：原模型基线推理

目标：保存本地 **Qwen3.5-4B 尚未加载 LoRA adapter 时**的输出，后续用相同提示词和推理参数比较微调效果。这里的“原模型”不是 `Qwen3.5-4B-Base`。

选择 **Python (Text LoRA - Qwen)** kernel。第一阶段已通过后，按顺序执行：预览固定提示词 → dry run → 单条真实推理 → 10 条完整基线 → 人工检查。

本 Notebook 不安装依赖、不下载模型、不启动训练。标注为“真实推理”的 Cell 会加载本地权重到 GPU；每次推理在独立子进程中完成，结束后释放该进程的模型显存。

In [1]:
import json
import subprocess
import sys
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    path
    for candidate in candidates
    for path in (candidate, candidate / "text-lora")
    if (path / "scripts" / "run_inference.py").is_file()
    and (path / "configs" / "models.toml").is_file()
)
sys.path.insert(0, str(project_root / "scripts"))
print("项目：", project_root)
print("解释器：", sys.executable)

项目： /home/muyi086/github/test_LLM/text-lora
解释器： /home/muyi086/github/test_LLM/text-lora/.venv/bin/python3


In [2]:
from check_environment import check_python, load_model_config
from run_inference import load_inference_config, load_prompts, prompts_fingerprint

check_python()
MODEL_PROFILE = "qwen3_5_4b"
model = load_model_config(MODEL_PROFILE)
settings = load_inference_config()
cases = load_prompts()
print("模型：", model.name)
print("提示词数量：", len(cases))
print("提示词 SHA256：", prompts_fingerprint(cases))
print("参数：", settings)
for case in cases:
    print(case["id"], case["category"])

Python：3.12.13
解释器：/home/muyi086/github/test_LLM/text-lora/.venv/bin/python3
模型： Qwen3.5-4B
提示词数量： 10
提示词 SHA256： 10afb61ffb990de1bb9de7f08198711c7393d7c7f4511737a5f98527ea5aa888
参数： InferenceConfig(max_tokens=384, temperature=0.0, top_p=1.0, repetition_penalty=1.0, seed=42, enable_thinking=False, device='cuda:0', attn_impl='sdpa')
tts_01_knock tts_rewrite
tts_02_dialogue tts_rewrite
tts_03_monologue tts_rewrite
tts_04_three_roles tts_rewrite
tts_05_phone tts_rewrite
tts_06_action tts_rewrite
tts_07_quiet tts_rewrite
tts_08_ambiguity tts_rewrite
control_01_polish general_control
control_02_explain general_control


## 先理解输入和检查标准

每条 `messages` 只有 system/user 输入，没有 assistant 目标答案。8 条改写任务要求 `[角色][语气]正文`，2 条普通任务用来观察 LoRA 是否让模型对所有任务都输出剧本格式。

这些提示词是固定测试集，后续教学训练数据应另写故事片段。基线保存后先保留提示词与 `configs/inference.toml`，两次对照时核对记录中的 SHA256 和推理参数。

In [3]:
print(json.dumps(cases[0]["messages"], ensure_ascii=False, indent=2))
print("人工检查：")
for criterion in cases[0]["review_criteria"]:
    print("-", criterion)

[
  {
    "role": "system",
    "content": "你是一名有声剧导演。将给定小说片段改写为多角色演绎稿。每行采用[角色][语气]正文，角色沿用原文姓名，无名人物使用原文称呼，叙述使用旁白。保留原文事实、人物关系、动作和悬疑信息，不补充原文没有的情节。只输出演绎稿，不解释改写过程。"
  },
  {
    "role": "user",
    "content": "改写下面的文本：\n走廊尽头传来三下敲门声。林舟停下脚步，低声问：谁在那里？门后没有回答，只有一阵拖动椅子的声音。"
  }
]
人工检查：
- 保留三下敲门和拖椅子的声音
- 林舟的问话与旁白分开
- 没有补充门后人物身份


## 运行方式与 dry run

复用 `scripts/run_inference.py`，由当前项目 kernel 的 Python 启动子进程。输出逐行显示，模型加载及推理可能需要等待。下面的 dry run 只校验提示词、参数和模型文件，不加载权重。

In [4]:
def run_command(command):
    with subprocess.Popen(
        command,
        cwd=project_root,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            return_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


command = [
    sys.executable,
    "-u",
    str(project_root / "scripts" / "run_inference.py"),
    "--model",
    MODEL_PROFILE,
]
run_command([*command, "--dry-run"])

模型：Qwen3.5-4B
本地路径：/home/muyi086/hf-mirror/Qwen/Qwen3.5-4B
Adapter：None
提示词：10 条，SHA256=10afb61ffb990de1bb9de7f08198711c7393d7c7f4511737a5f98527ea5aa888
推理参数：{'max_tokens': 384, 'temperature': 0.0, 'top_p': 1.0, 'repetition_penalty': 1.0, 'seed': 42, 'enable_thinking': False, 'device': 'cuda:0', 'attn_impl': 'sdpa'}
输出目录：/home/muyi086/github/test_LLM/text-lora/outputs/base-20261008T012200_395008Z
模型：Qwen3.5-4B
路径：/home/muyi086/hf-mirror/Qwen/Qwen3.5-4B
架构：['Qwen3_5ForConditionalGeneration']
权重：2 个文件，8.68 GiB
检查了文件存在性和非空状态；未进行权重校验和验证。
Dry run 通过。未加载模型、未创建输出目录。


## 单条真实推理：此 Cell 开始加载权重

先运行第一条，确认模型能生成并保存结果。当前设置为 BF16、单条推理、`temperature=0`、`enable_thinking=false`、最多生成 384 个 token。启动前结束占用显存的其他模型服务。

成功应显示改写结果和保存路径；如果出现 Traceback，先处理该错误，再运行完整基线。运行中断可用 Notebook 的停止按钮，子进程会被终止。

In [5]:
smoke_dir = (
    project_root / "outputs" / ("base-smoke-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ"))
)
run_command([*command, "--limit", "1", "--output-dir", str(smoke_dir)])
smoke_metadata = json.loads((smoke_dir / "metadata.json").read_text())
assert smoke_metadata["status"] == "complete"
print("单条推理通过：", smoke_dir)

模型：Qwen3.5-4B
本地路径：/home/muyi086/hf-mirror/Qwen/Qwen3.5-4B
Adapter：None
提示词：1 条，SHA256=6321139b0083237f1ca705acecddfbeab37025114d76dbaab9f0bfed664b89a0
推理参数：{'max_tokens': 384, 'temperature': 0.0, 'top_p': 1.0, 'repetition_penalty': 1.0, 'seed': 42, 'enable_thinking': False, 'device': 'cuda:0', 'attn_impl': 'sdpa'}
输出目录：/home/muyi086/github/test_LLM/text-lora/outputs/base-smoke-20261008T012347_943256Z
模型：Qwen3.5-4B
路径：/home/muyi086/hf-mirror/Qwen/Qwen3.5-4B
架构：['Qwen3_5ForConditionalGeneration']
权重：2 个文件，8.68 GiB
检查了文件存在性和非空状态；未进行权重校验和验证。
Python：3.12.13
解释器：/home/muyi086/github/test_LLM/text-lora/.venv/bin/python3
PyTorch：2.11.0+cu130
CUDA runtime：13.0
GPU：NVIDIA GeForce RTX 4070 Ti SUPER
BF16：True
显存：空闲 14.73 / 总量 15.99 GiB
开始将本地 BF16 权重加载到 GPU……
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[INFO:swift] attn_impl: sdpa
[INFO:swift] Setting max_ratio: 200. You can adjust this hyperparameter throu

## 完整基线：10 条提示词

下面重新加载一次模型，逐条生成 10 个结果。会建立新的输出目录，保留前面的单条记录。`results.jsonl` 每行保存原始输入、输出、结束原因和 token 用量；`metadata.json` 保存模型路径、依赖版本、推理参数、提示词哈希和运行状态。

`finish_reason=length` 表示输出触及上限，需标记为可能截断。若修改 `max_tokens`，后续也应使用相同设置重新生成完整基线。

In [6]:
baseline_dir = (
    project_root / "outputs" / ("base-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ"))
)
run_command([*command, "--output-dir", str(baseline_dir)])
print("保留这次完整基线目录：", baseline_dir)

模型：Qwen3.5-4B
本地路径：/home/muyi086/hf-mirror/Qwen/Qwen3.5-4B
Adapter：None
提示词：10 条，SHA256=10afb61ffb990de1bb9de7f08198711c7393d7c7f4511737a5f98527ea5aa888
推理参数：{'max_tokens': 384, 'temperature': 0.0, 'top_p': 1.0, 'repetition_penalty': 1.0, 'seed': 42, 'enable_thinking': False, 'device': 'cuda:0', 'attn_impl': 'sdpa'}
输出目录：/home/muyi086/github/test_LLM/text-lora/outputs/base-20261008T012651_918242Z
模型：Qwen3.5-4B
路径：/home/muyi086/hf-mirror/Qwen/Qwen3.5-4B
架构：['Qwen3_5ForConditionalGeneration']
权重：2 个文件，8.68 GiB
检查了文件存在性和非空状态；未进行权重校验和验证。
Python：3.12.13
解释器：/home/muyi086/github/test_LLM/text-lora/.venv/bin/python3
PyTorch：2.11.0+cu130
CUDA runtime：13.0
GPU：NVIDIA GeForce RTX 4070 Ti SUPER
BF16：True
显存：空闲 14.73 / 总量 15.99 GiB
开始将本地 BF16 权重加载到 GPU……
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[INFO:swift] attn_impl: sdpa
[INFO:swift] Setting max_ratio: 200. You can adjust this hyperparameter through th

## 确认记录完整，并写下观察

先核对 10 条记录与 `status=complete`，再逐条阅读输出。不是每条结果都必须满足人工检查项目；未达到要求的地方正是后续微调要观察的内容。

接下来的 Cell 建立 `review.md`，将观察记录填到该文件中。基线结果不要手工改写。

In [7]:
metadata = json.loads((baseline_dir / "metadata.json").read_text())
results = [
    json.loads(line)
    for line in (baseline_dir / "results.jsonl").read_text().splitlines()
    if line.strip()
]
assert metadata["status"] == "complete"
assert metadata["completed_count"] == len(cases) == len(results)
assert metadata["prompts_sha256"] == prompts_fingerprint(cases)
assert metadata["inference"] == asdict(settings)
for result in results:
    print(result["id"], result["finish_reason"], result["usage"])
print("完整记录：", baseline_dir / "results.jsonl")

notes = ["# 原模型基线观察", "", f"提示词 SHA256：{metadata['prompts_sha256']}", ""]
for result in results:
    notes.extend([f"## {result['id']}", ""])
    notes.extend(f"- [ ] {criterion}" for criterion in result["review_criteria"])
    notes.extend(["", f"结束原因：{result['finish_reason']}", "", "观察：", "", ""])
review_path = baseline_dir / "review.md"
if not review_path.exists():
    review_path.write_text("\n".join(notes), encoding="utf-8")
print("人工笔记：", review_path)
print("下一阶段：准备独立的教学训练集与验证集，再做 LoRA。")

tts_01_knock stop {'prompt_tokens': 125, 'completion_tokens': 44, 'total_tokens': 169}
tts_02_dialogue stop {'prompt_tokens': 125, 'completion_tokens': 43, 'total_tokens': 168}
tts_03_monologue stop {'prompt_tokens': 127, 'completion_tokens': 75, 'total_tokens': 202}
tts_04_three_roles stop {'prompt_tokens': 130, 'completion_tokens': 64, 'total_tokens': 194}
tts_05_phone stop {'prompt_tokens': 130, 'completion_tokens': 84, 'total_tokens': 214}
tts_06_action stop {'prompt_tokens': 130, 'completion_tokens': 54, 'total_tokens': 184}
tts_07_quiet stop {'prompt_tokens': 129, 'completion_tokens': 76, 'total_tokens': 205}
tts_08_ambiguity stop {'prompt_tokens': 127, 'completion_tokens': 61, 'total_tokens': 188}
control_01_polish stop {'prompt_tokens': 70, 'completion_tokens': 14, 'total_tokens': 84}
control_02_explain stop {'prompt_tokens': 54, 'completion_tokens': 64, 'total_tokens': 118}
完整记录： /home/muyi086/github/test_LLM/text-lora/outputs/base-20261008T012651_918242Z/results.jsonl
人工笔记： /